# 04 SOKE Gemma Dataset Inspection

Inspect the SOKE-style LM rows used by the Gemma 3 270M LoRA training notebook. The checks here are intentionally about formulation: source split counts, rendered instruction prompts, flattened motion-token targets, and atomic tokenizer behavior.

In [ ]:
from pathlib import Path
import json
import os
import random
import sys

BUNDLE_ROOT = Path(os.environ.get('SOKE_GEMMA_BUNDLE_ROOT', '.')).resolve()
if not (BUNDLE_ROOT / 'scripts').exists():
    BUNDLE_ROOT = Path('/home/cem/tez/exp/new_data/04_soke_gemma3_270m_lora_lm')
CODE_ROOT = Path(os.environ.get('SOKE_GEMMA_CODE_ROOT', str(BUNDLE_ROOT / 'outputs' / 'soke_motion_codes')))
INSTRUCTIONS_ROOT = Path(os.environ.get('SOKE_GEMMA_INSTRUCTIONS_ROOT', str(BUNDLE_ROOT / 'instructions')))
SPLIT = os.environ.get('SOKE_GEMMA_INSPECT_SPLIT', 'train')
STAGE = os.environ.get('SOKE_GEMMA_STAGE', 'lm_pretrain')
N_RANDOM = int(os.environ.get('SOKE_GEMMA_INSPECT_N', '10'))
SEED = int(os.environ.get('SOKE_GEMMA_SEED', '42'))
TOKENIZER_NAME = os.environ.get('SOKE_GEMMA_TOKENIZER', 'google/gemma-3-270m')

sys.path.insert(0, str(BUNDLE_ROOT / 'scripts'))
print({'BUNDLE_ROOT': str(BUNDLE_ROOT), 'CODE_ROOT': str(CODE_ROOT), 'SPLIT': SPLIT, 'STAGE': STAGE})

In [ ]:
from soke_gemma_data import SokeGemmaCausalDataset, load_codecs, load_instructions, read_jsonl
import pandas as pd

code_path = CODE_ROOT / f'{SPLIT}_soke_motion_codes.jsonl'
if not code_path.exists():
    raise FileNotFoundError(f'Missing code JSONL: {code_path}')
rows = read_jsonl(code_path)
codecs = load_codecs(CODE_ROOT)
inst_path = INSTRUCTIONS_ROOT / ('template_pretrain.json' if STAGE == 'lm_pretrain' else 'template_instructions.json')
tasks = load_instructions(inst_path)
ds = SokeGemmaCausalDataset(rows, tasks, codecs, random_drop=False, seed=SEED)
print(json.dumps({
    'physical_rows': len(rows),
    'instruction_tasks': len(tasks),
    'logical_rows': len(ds),
    'codecs': codecs.to_json_dict(),
}, indent=2))

In [ ]:
df = pd.DataFrame(rows)
summary = df.groupby(['dataset', 'source_alias']).agg(
    clips=('clip_id', 'count'),
    code_len_mean=('code_len', 'mean'),
    code_len_median=('code_len', 'median'),
    code_len_min=('code_len', 'min'),
    code_len_max=('code_len', 'max'),
    frames_mean=('num_frames_soke', 'mean'),
).reset_index()
display(summary)
display(df[['clip_key', 'dataset', 'source_alias', 'clip_id', 'code_len', 'num_frames_soke', 'text']].sample(min(12, len(df)), random_state=SEED))

In [ ]:
rng = random.Random(SEED)
picks = [rng.randrange(len(ds)) for _ in range(min(N_RANDOM, len(ds)))]
sample_rows = []
for logical_idx in picks:
    sample = ds[logical_idx]
    sample_rows.append({
        'logical_idx': logical_idx,
        'clip_key': sample['clip_key'],
        'dataset': sample['dataset'],
        'source_alias': sample['source_alias'],
        'code_len': sample['code_len'],
        'target_token_count': sample['target_token_count'],
        'caption': sample['caption'],
        'prompt': sample['prompt'],
        'target_prefix': ' '.join(sample['target'].split()[:36]),
    })
samples_df = pd.DataFrame(sample_rows)
display(samples_df)

for row in sample_rows[:5]:
    print('=' * 120)
    print(json.dumps(row, indent=2, ensure_ascii=False))

In [ ]:
# Optional tokenizer audit. This verifies SOKE motion symbols are single added tokens.
RUN_TOKENIZER_AUDIT = os.environ.get('SOKE_GEMMA_RUN_TOKENIZER_AUDIT', '1') == '1'
if RUN_TOKENIZER_AUDIT:
    from transformers import AddedToken, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
    tokenizer.add_tokens([
        AddedToken(tok, lstrip=True, rstrip=False, normalized=False, special=False)
        for tok in codecs.added_tokens()
    ])
    prefixes = ('<motion_id_', '<hand_id_', '<rhand_id_')
    audits = []
    for logical_idx in picks[:min(8, len(picks))]:
        sample = ds[logical_idx]
        motion_text = ' '.join(tok for tok in sample['target'].split() if tok.startswith(prefixes))
        expected = len(motion_text.split())
        got = len(tokenizer(motion_text, add_special_tokens=False).input_ids)
        audits.append({'clip_key': sample['clip_key'], 'expected_motion_tokens': expected, 'tokenizer_ids': got, 'ok': expected == got})
    audit_df = pd.DataFrame(audits)
    display(audit_df)
    if not audit_df['ok'].all():
        raise RuntimeError('SOKE motion token atomicity audit failed')
else:
    print('Tokenizer audit skipped')